In [1]:
# Post-hoc recalibration on 2021-2023 (R-005).
# This is NOT the primary result (see R-004). It shows how much updating the
# model's risk level would help, using cross-validation inside 2021-2023 so the
# recalibration is always tested on people it was not fitted on.
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
pd.set_option("display.width", 200)

def find_project_root():
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "src" / "modeling.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find src/modeling.py from " + str(path))

sys.path.insert(0, str(find_project_root() / "src"))
import importlib
import modeling as m
importlib.reload(m)

dev = m.load("2017-2020")
val = m.load("2021-2023")
y_dev = dev[m.OUTCOME].to_numpy()
y_val = val[m.OUTCOME].to_numpy()

# Same final model as R-004 (deterministic, so identical predictions)
model_A = m.make_logistic(m.MODEL_A).fit(dev[m.MODEL_A], y_dev)
p = np.clip(model_A.predict_proba(val[m.MODEL_A])[:, 1], 1e-6, 1 - 1e-6)
lp = np.log(p / (1 - p))                       # original linear predictor (logit)
T_FIXED = 0.106                                # threshold from R-004 / D-013

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def fit_intercept(lp_train, y_train):
    """Calibration-in-the-large: new logit = a + original logit."""
    def loss(a):
        q = np.clip(sigmoid(a + lp_train), 1e-9, 1 - 1e-9)
        return -np.mean(y_train * np.log(q) + (1 - y_train) * np.log(1 - q))
    a = minimize_scalar(loss, bounds=(-5, 5), method="bounded").x
    return lambda lp_new: sigmoid(a + lp_new)

def fit_logistic(lp_train, y_train):
    """Logistic recalibration: new logit = a + b * original logit."""
    lr = LogisticRegression(C=np.inf).fit(lp_train.reshape(-1, 1), y_train)
    return lambda lp_new: lr.predict_proba(lp_new.reshape(-1, 1))[:, 1]

METHODS = {
    "Original (no update)": None,
    "Intercept update": fit_intercept,
    "Intercept + slope update": fit_logistic,
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=m.RANDOM_STATE)
rows, oof = [], {name: np.zeros(len(y_val)) for name in METHODS}
for i, (train, test) in enumerate(cv.split(lp, y_val)):
    for name, fit in METHODS.items():
        q = p[test] if fit is None else fit(lp[train], y_val[train])(lp[test])
        if i < 5:                              # first repeat -> one out-of-fold set
            oof[name][test] = q
        r = m.metrics(y_val[test], q)
        r.update(m.at_threshold(y_val[test], q, T_FIXED))
        rows.append({"method": name, **r})

res = pd.DataFrame(rows).groupby("method", sort=False)[
    ["roc_auc", "brier", "cal_slope", "obs_exp", "sensitivity", "specificity", "referred_pct"]
].mean()
print("=== Validation 2021-2023, cross-validated within 2021-2023 (5-fold x 10) ===")
print(f"Fixed risk threshold for referral: {T_FIXED}\n")
print(res.round(3), "\n")

# Calibration by decile, out-of-fold (first repeat)
deciles = pd.qcut(p, 10, labels=False, duplicates="drop")
table = pd.DataFrame({"observed %": pd.Series(y_val).groupby(deciles).mean() * 100})
for name in METHODS:
    table[f"{name} %"] = pd.Series(oof[name]).groupby(deciles).mean() * 100
print("=== Predicted vs observed by decile of original risk (out-of-fold) ===")
print(table.round(1))

=== Validation 2021-2023, cross-validated within 2021-2023 (5-fold x 10) ===
Fixed risk threshold for referral: 0.106

                          roc_auc  brier  cal_slope  obs_exp  sensitivity  specificity  referred_pct
method                                                                                              
Original (no update)        0.709  0.095      0.834    0.699        0.819        0.480        55.144
Intercept update            0.709  0.091      0.835    1.001        0.688        0.621        41.196
Intercept + slope update    0.709  0.091      1.017    1.001        0.710        0.601        43.164 

=== Predicted vs observed by decile of original risk (out-of-fold) ===
   observed %  Original (no update) %  Intercept update %  Intercept + slope update %
0         1.1                     2.6                 1.7                         2.4
1         3.1                     3.9                 2.5                         3.4
2         3.9                     5.6        